# BirdCLEF+ 2026 — Kaggle CPU Submission Notebook

**Hard constraint:** all inference must finish within **85 minutes** (Kaggle 90-min cap with 5-min safety margin).  
**Self-contained:** no imports from src/ — all helpers copied inline.  
**Run order:** Cells 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8  

Required Kaggle inputs (attach before running):
- Competition data: `birdclef-2026`
- Trained models dataset: `{username}/birdclef2026-trained-models`
- Perch v2 CPU model: `google/perch/tfSavedModel/perch_v2_cpu/1`  *(bold mode only)*
- YAMNet model: `google/yamnet/tfSavedModel/yamnet/1`  *(bold mode only)*

In [ ]:
# ============================================================
# CELL 0 — SUBMISSION CONFIG (edit only this cell)
# ============================================================

SUBMISSION_MODE = 'conservative'  # 'conservative' | 'bold'
# conservative: B0 ONNX + EfficientAT ONNX + group-weighted rank average
# bold:         adds Perch embedding head + YAMNet embedding head (slower)

KAGGLE_MODELS_DIR = '/kaggle/input/birdclef2026-trained-models'
TEST_DIR          = '/kaggle/input/birdclef-2026/test_soundscapes'
TAXONOMY_PATH     = '/kaggle/input/birdclef-2026/taxonomy.csv'
SAMPLE_SUB_PATH   = '/kaggle/input/birdclef-2026/sample_submission.csv'

# Perch / YAMNet model paths (used only in bold mode)
PERCH_PATH  = '/kaggle/input/perch/tensorflow2/perch_v2_cpu/1'
YAMNET_PATH = '/kaggle/input/yamnet/tensorflow2/yamnet/1'

# Critical constants — identical to src/ and Colab training notebook
SR          = 32000
N_FFT       = 2048
HOP_LENGTH  = 512
N_MELS      = 128
F_MIN       = 20
F_MAX       = 16000
WINDOW_SECS = 5
INPUT_H     = 224
INPUT_W     = 224
NUM_CLASSES = 234

BATCH_SIZE  = 8   # windows processed per inference batch

print(f'SUBMISSION_MODE: {SUBMISSION_MODE}')

In [ ]:
# ============================================================
# CELL 1 — TIMER SETUP
# ============================================================
import time

START_TIME     = time.time()
BUDGET_SECONDS = 85 * 60  # 85-minute hard budget

def time_remaining():
    return BUDGET_SECONDS - (time.time() - START_TIME)

def check_budget(label=''):
    rem = time_remaining()
    elapsed = time.time() - START_TIME
    tag = f'[{label}] ' if label else ''
    print(f'{tag}Elapsed: {elapsed/60:.1f} min | Remaining: {rem/60:.1f} min')
    if rem < 300:  # less than 5 min left → abort cleanly
        raise TimeoutError(
            f'Budget nearly exhausted at: {label} '
            f'(remaining={rem:.0f}s). Saving partial submission.')

print(f'Timer started. Budget: {BUDGET_SECONDS/60:.0f} min')
check_budget('timer init')

In [ ]:
# ============================================================
# CELL 2 — PACKAGE INSTALLS + IMPORTS
# ============================================================
import subprocess as _sub
_sub.run(['pip', 'install', '-q', 'onnxruntime', 'librosa', 'opencv-python-headless'],
         check=False)

import os, warnings
import numpy as np
import pandas as pd
import librosa
import cv2
import onnxruntime as ort
from pathlib import Path
from tqdm.auto import tqdm
warnings.filterwarnings('ignore')

# TensorFlow only needed in bold mode
if SUBMISSION_MODE == 'bold':
    import tensorflow as tf
    import tensorflow_hub as hub
    print(f'tensorflow: {tf.__version__}')

print(f'onnxruntime: {ort.__version__}')
check_budget('imports done')

In [ ]:
# ============================================================
# CELL 3 — LOAD TAXONOMY + SUBMISSION TEMPLATE
# ============================================================
taxonomy   = pd.read_csv(TAXONOMY_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

TARGET_COLUMNS = list(sample_sub.columns)          # ['row_id', sp1, sp2, ...]
SPECIES_IDS    = TARGET_COLUMNS[1:]                 # 234 species in submission order

assert len(SPECIES_IDS) == 234, \
    f'FAILED: expected 234 species columns, got {len(SPECIES_IDS)}'

# ── Build bird / non-bird masks from taxonomy (not hardcoded) ────────
# Birds: class_name == 'Aves'; species identified by primary_label column
taxonomy['primary_label'] = taxonomy['primary_label'].astype(str)
bird_species_set = set(taxonomy.loc[taxonomy['class_name'] == 'Aves', 'primary_label'])

bird_mask    = np.array([1 if s in bird_species_set else 0 for s in SPECIES_IDS], dtype=np.float32)
nonbird_mask = 1.0 - bird_mask

print(f'Bird species  : {int(bird_mask.sum())}  (expected 162)')
print(f'Non-bird      : {int(nonbird_mask.sum())}  (expected 72)')
assert int(bird_mask.sum()) == 162, \
    f'FAILED: bird species count {int(bird_mask.sum())}, expected 162'
assert int(nonbird_mask.sum()) == 72, \
    f'FAILED: non-bird species count {int(nonbird_mask.sum())}, expected 72'

# Species index mapping
sp2idx = {s: i for i, s in enumerate(SPECIES_IDS)}
print(f'Total submission rows: {len(sample_sub)}')
check_budget('taxonomy loaded')

In [ ]:
# ============================================================
# CELL 4 — LOAD MODELS
# ============================================================
t_load = time.time()
print(f"\n{'='*50}\n[CELL 4] Loading models\n{'='*50}")

models_dir = Path(KAGGLE_MODELS_DIR)

# ── Helper: load ONNX session with graceful fallback ─────────────────
def load_onnx_session(path, name='model'):
    p = Path(path)
    if not p.exists():
        print(f'  WARNING: {name} not found at {path} — skipping')
        return None
    sess = ort.InferenceSession(
        str(p), providers=['CPUExecutionProvider']
    )
    # Warm-up + shape check
    inp_name = sess.get_inputs()[0].name
    inp_shape = sess.get_inputs()[0].shape  # may contain dynamic dims
    # Build dummy with correct static dims (replace dynamic/None with 1)
    static_shape = [d if isinstance(d, int) and d > 0 else 1 for d in inp_shape]
    dummy = np.random.randn(*static_shape).astype(np.float32)
    out = sess.run(None, {inp_name: dummy})[0]
    assert out.shape[-1] == 234, \
        f'FAILED: {name} output last-dim {out.shape[-1]}, expected 234'
    print(f'  {name}: loaded OK  (input={inp_shape}, output={out.shape})')
    return sess

# ── 1. B0 ONNX (birds specialist; covers all 234 classes) ────────────
b0_session = load_onnx_session(models_dir / 'b0_fold0.onnx', 'B0')

# ── 2. EfficientAT ONNX (strongest on non-birds) ─────────────────────
eat_session = load_onnx_session(models_dir / 'efficientAT_fold0.onnx', 'EfficientAT')

# At least one mel-spec model must be present to proceed
mel_sessions = [s for s in [b0_session, eat_session] if s is not None]
assert mel_sessions, \
    'FAILED: no mel-spectrogram ONNX models found. Attach the trained-models dataset.'

# ── 3. Embedding heads + base models (bold mode only) ─────────────────
head_perch_session  = None
head_yamnet_session = None
perch_model         = None
yamnet_model        = None

if SUBMISSION_MODE == 'bold':
    head_perch_session  = load_onnx_session(models_dir / 'head_perch.onnx',  'HeadPerch')
    head_yamnet_session = load_onnx_session(models_dir / 'head_yamnet.onnx', 'HeadYAMNet')

    perch_path = Path(PERCH_PATH)
    if perch_path.exists():
        perch_model = tf.saved_model.load(str(perch_path))
        print(f'  Perch model loaded from {perch_path}')
    else:
        print(f'  WARNING: Perch model not found at {perch_path}')

    yamnet_path = Path(YAMNET_PATH)
    if yamnet_path.exists():
        yamnet_model = hub.load(str(yamnet_path))
        print(f'  YAMNet model loaded from {yamnet_path}')
    else:
        print(f'  WARNING: YAMNet model not found at {yamnet_path}')

# ── 4. Prior fallback calibration ────────────────────────────────────
prior_path = models_dir / 'species_priors.npy'
species_priors = np.load(str(prior_path)) if prior_path.exists() else None
if species_priors is not None:
    assert species_priors.shape == (234,), \
        f'FAILED: species_priors shape {species_priors.shape}, expected (234,)'
    print(f'  species_priors loaded (mean={species_priors.mean():.4f})')
else:
    print('  species_priors.npy not found — prior fallback disabled')

print(f'All models loaded in {time.time()-t_load:.1f}s')
check_budget('models loaded')

In [ ]:
# ============================================================
# CELL 5 — MEL SPECTROGRAM FUNCTION
# Verbatim copy of src/ensemble.py::compute_mel — must stay byte-identical to training.
# ============================================================

def compute_melspec(waveform, sr=32000):
    """
    Convert a 5-second mono waveform to (3, 224, 224) float32.
    Copied verbatim from src/ensemble.py — do NOT modify independently.
    Params: n_fft=2048, hop_length=512, n_mels=128, fmin=20, fmax=16000.
    Normalization: mel_db / 80.0 clipped to [-1, 1].
    """
    mel = librosa.feature.melspectrogram(
        y=waveform, sr=sr,
        n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS,
        fmin=F_MIN, fmax=F_MAX,
    )
    mel_db = librosa.power_to_db(mel, ref=np.max)
    mel_db = mel_db / 80.0            # normalize to ~[-1, 1]
    mel_db = np.clip(mel_db, -1.0, 1.0)
    mel_resized = cv2.resize(mel_db, (INPUT_W, INPUT_H))   # (224, 224)
    mel_3ch = np.stack([mel_resized, mel_resized, mel_resized], axis=0)
    return mel_3ch.astype(np.float32)  # (3, 224, 224)


# ── Self-test ────────────────────────────────────────────────────────
test_wav = np.random.randn(SR * WINDOW_SECS).astype(np.float32)
test_mel = compute_melspec(test_wav)
assert test_mel.shape == (3, 224, 224), \
    f'FAILED: compute_melspec shape {test_mel.shape}, expected (3, 224, 224)'
assert test_mel.dtype == np.float32, \
    f'FAILED: compute_melspec dtype {test_mel.dtype}, expected float32'
assert test_mel.min() >= -1.0 and test_mel.max() <= 1.0, \
    f'FAILED: compute_melspec values out of [-1,1]: [{test_mel.min():.2f}, {test_mel.max():.2f}]'
print(f'compute_melspec self-test OK — shape={test_mel.shape}, '
      f'range=[{test_mel.min():.3f}, {test_mel.max():.3f}]')
check_budget('melspec fn ready')

In [ ]:
# ============================================================
# CELL 6 — RANK-AVERAGE ENSEMBLE FUNCTION
# ============================================================

def rank_average(score_arrays):
    """
    Rank-average an arbitrary number of score matrices.
    score_arrays : list of np.ndarray, each shape (N, 234)
    Returns      : np.ndarray (N, 234) rank-averaged scores in [1, N]
    """
    from scipy.stats import rankdata
    ranked = [np.apply_along_axis(rankdata, 0, s) for s in score_arrays]
    return np.mean(ranked, axis=0)


# Per-model group weights (bird vs. non-bird columns)
# Determined from soundscape validation; B0 is stronger on birds,
# EfficientAT / YAMNet head on non-birds.
# Format: [(session, bird_weight, nonbird_weight), ...]
def _build_weight_matrix(sessions_with_weights, n_windows):
    """Pre-build per-column weight arrays for each model output."""
    weights = []
    for _, bw, nbw in sessions_with_weights:
        w = np.where(bird_mask.astype(bool), bw, nbw).astype(np.float32)  # (234,)
        weights.append(np.tile(w, (n_windows, 1)))  # (N, 234)
    return weights  # list of (N, 234)


def _run_mel_session(sess, mel_batch):
    """Run a mel-spec ONNX session, returning (B, 234) logits."""
    if sess is None:
        return np.zeros((mel_batch.shape[0], NUM_CLASSES), dtype=np.float32)
    inp = sess.get_inputs()[0].name
    return sess.run(None, {inp: mel_batch})[0].astype(np.float32)


def _run_embedding_head(base_model_fn, head_sess, waveform_batch, emb_dim):
    """Extract embeddings via base_model_fn and run through ONNX head."""
    if head_sess is None or base_model_fn is None:
        return None
    B = waveform_batch.shape[0]
    embs = np.zeros((B, emb_dim), dtype=np.float32)
    for i, wav in enumerate(waveform_batch):
        try:
            embs[i] = base_model_fn(wav)
        except Exception:
            pass
    inp = head_sess.get_inputs()[0].name
    return head_sess.run(None, {inp: embs})[0].astype(np.float32)  # (B, 234)


def _get_perch_embedding(wav_np):
    if perch_model is None:
        return np.zeros(1280, dtype=np.float32)
    wt = tf.constant(wav_np, dtype=tf.float32)
    result = perch_model.infer_tf(wt[tf.newaxis])
    return result['embedding'].numpy().mean(axis=1).squeeze().astype(np.float32)


def _get_yamnet_embedding(wav_np):
    if yamnet_model is None:
        return np.zeros(1024, dtype=np.float32)
    scores, embeddings, _ = yamnet_model(wav_np)
    return embeddings.numpy().mean(axis=0).astype(np.float32)


def ensemble_predict(mel_batch, waveform_batch=None):
    """
    mel_batch      : np.ndarray (B, 3, 224, 224) float32
    waveform_batch : np.ndarray (B, 160000) float32, only used in 'bold' mode
    Returns        : np.ndarray (B, 234) — rank-averaged ensemble scores
    """
    B = mel_batch.shape[0]
    score_arrays = []
    model_weights = []  # bird_w, nonbird_w per model

    # ── B0 (strongest on birds) ─────────────────────────────────────
    b0_out = _run_mel_session(b0_session, mel_batch)
    if b0_session is not None:
        score_arrays.append(b0_out)
        model_weights.append((0.50, 0.20))  # (bird_weight, nonbird_weight)

    # ── EfficientAT (strongest on non-birds) ────────────────────────
    eat_out = _run_mel_session(eat_session, mel_batch)
    if eat_session is not None:
        score_arrays.append(eat_out)
        model_weights.append((0.30, 0.50))

    # ── Perch head (bird specialist, bold only) ──────────────────────
    if SUBMISSION_MODE == 'bold' and waveform_batch is not None:
        perch_out = _run_embedding_head(_get_perch_embedding, head_perch_session,
                                        waveform_batch, emb_dim=1280)
        if perch_out is not None:
            score_arrays.append(perch_out)
            model_weights.append((0.20, 0.10))

        # ── YAMNet head (non-bird specialist, bold only) ─────────────
        yamnet_out = _run_embedding_head(_get_yamnet_embedding, head_yamnet_session,
                                         waveform_batch, emb_dim=1024)
        if yamnet_out is not None:
            score_arrays.append(yamnet_out)
            model_weights.append((0.10, 0.30))

    if not score_arrays:
        return np.full((B, NUM_CLASSES), 0.5, dtype=np.float32)

    # ── Rank-average all available models ───────────────────────────
    ensemble = rank_average(score_arrays)  # (B, 234) rank scores

    # ── Group routing: re-weight per bird/non-bird columns ───────────
    if len(score_arrays) > 1 and len(model_weights) == len(score_arrays):
        # Compute weighted sum of sigmoid outputs per column group
        from scipy.special import expit as sigmoid
        weighted = np.zeros((B, NUM_CLASSES), dtype=np.float32)
        total_w  = np.zeros(NUM_CLASSES, dtype=np.float32)
        for raw_scores, (bw, nbw) in zip(score_arrays, model_weights):
            probs = sigmoid(raw_scores).astype(np.float32)
            col_w = np.where(bird_mask.astype(bool), bw, nbw).astype(np.float32)
            weighted += probs * col_w[np.newaxis, :]
            total_w  += col_w
        ensemble = weighted / np.clip(total_w[np.newaxis, :], 1e-8, None)

    return ensemble.astype(np.float32)  # (B, 234)


# ── Self-test ────────────────────────────────────────────────────────
dummy_mel = np.random.randn(2, 3, 224, 224).astype(np.float32)
dummy_wav = np.random.randn(2, SR * WINDOW_SECS).astype(np.float32)
test_out = ensemble_predict(dummy_mel, dummy_wav)
assert test_out.shape == (2, 234), \
    f'FAILED: ensemble_predict shape {test_out.shape}, expected (2, 234)'
print(f'ensemble_predict self-test OK — shape={test_out.shape}')
check_budget('ensemble fn ready')

In [ ]:
# ============================================================
# CELL 7 — MAIN INFERENCE LOOP
# ============================================================
print(f"\n{'='*50}\n[CELL 7] Main inference\n{'='*50}")

SMOOTH_KERNEL = np.array([0.1, 0.2, 0.4, 0.2, 0.1], dtype=np.float32)  # temporal smoother


def temporal_smooth_file(preds_list):
    """
    Apply [0.1, 0.2, 0.4, 0.2, 0.1] weighted smoother along the time axis.
    preds_list : list of np.ndarray (234,)
    Returns    : list of np.ndarray (234,) smoothed
    """
    if len(preds_list) <= 1:
        return preds_list
    mat = np.stack(preds_list, axis=0)  # (T, 234)
    smoothed = np.apply_along_axis(
        lambda x: np.convolve(x, SMOOTH_KERNEL, mode='same'), axis=0, arr=mat
    )
    return [smoothed[i] for i in range(len(preds_list))]


test_files = sorted(Path(TEST_DIR).glob('*.ogg'))
print(f'Test soundscape files: {len(test_files)}')

if len(test_files) == 0:
    print('WARNING: no test files found at TEST_DIR — check paths.')

submission_rows = []    # accumulated {row_id: str, sp1: float, ...}
window_len      = WINDOW_SECS * SR  # 160000

for file_idx, soundscape_path in enumerate(tqdm(test_files, desc='Soundscapes')):
    # Emergency exit if budget is nearly exhausted
    if time_remaining() < 300:
        print(f'WARNING: time budget reached at file {file_idx}/{len(test_files)}. '
              f'Stopping inference early.')
        break

    # ── Load full soundscape ────────────────────────────────────────
    try:
        waveform, _ = librosa.load(str(soundscape_path), sr=SR, mono=True)
    except Exception as e:
        print(f'  ERROR loading {soundscape_path.name}: {e}')
        continue

    # ── Slice into 5-second non-overlapping windows ─────────────────
    n_windows = max(1, len(waveform) // window_len)
    file_mels, file_wavs, file_row_ids = [], [], []

    for win_idx in range(n_windows):
        start = win_idx * window_len
        chunk = waveform[start: start + window_len]
        if len(chunk) < window_len:
            chunk = np.pad(chunk, (0, window_len - len(chunk)))
        chunk = chunk.astype(np.float32)

        end_time = (win_idx + 1) * WINDOW_SECS
        row_id   = f'{soundscape_path.stem}_{end_time}'

        try:
            mel = compute_melspec(chunk)
        except Exception:
            mel = np.zeros((3, INPUT_H, INPUT_W), dtype=np.float32)

        file_mels.append(mel)
        file_wavs.append(chunk)
        file_row_ids.append(row_id)

    # ── Batch inference over this file's windows ────────────────────
    file_preds = []
    for batch_start in range(0, len(file_mels), BATCH_SIZE):
        batch_end  = min(batch_start + BATCH_SIZE, len(file_mels))
        mel_batch  = np.stack(file_mels[batch_start:batch_end]).astype(np.float32)
        wav_batch  = np.stack(file_wavs[batch_start:batch_end]).astype(np.float32)

        preds = ensemble_predict(mel_batch, wav_batch)  # (B, 234)
        for i in range(len(mel_batch)):
            file_preds.append(preds[i])

    # ── Temporal smoothing (within this file) ───────────────────────
    file_preds_smooth = temporal_smooth_file(file_preds)

    # ── Normalise to [0, 1] ─────────────────────────────────────────
    for i, (pred, row_id) in enumerate(zip(file_preds_smooth, file_row_ids)):
        p = pred.astype(np.float32)
        lo, hi = p.min(), p.max()
        if hi > lo:
            p = (p - lo) / (hi - lo + 1e-8)
        else:
            p = np.full_like(p, 0.5)

        # Prior fallback: floor zero-clip species at 30% of their prior
        if species_priors is not None:
            p = np.maximum(p, species_priors * 0.3)

        row = {'row_id': row_id}
        for j, sp in enumerate(SPECIES_IDS):
            row[sp] = float(p[j])
        submission_rows.append(row)

    # Progress report every 50 files
    if (file_idx + 1) % 50 == 0:
        check_budget(f'file {file_idx+1}/{len(test_files)}')

check_budget('inference complete')
print(f'Collected {len(submission_rows)} prediction rows.')

In [ ]:
# ============================================================
# CELL 8 — GENERATE SUBMISSION CSV
# ============================================================
print(f"\n{'='*50}\n[CELL 8] Build submission CSV\n{'='*50}")

# ── Build DataFrame from collected rows ──────────────────────────────
if submission_rows:
    submission_df = pd.DataFrame(submission_rows)
else:
    print('WARNING: no prediction rows — filling entire submission with 0.5')
    submission_df = pd.DataFrame(columns=TARGET_COLUMNS)

# ── Fill any missing row_ids with 0.5 (fallback) ─────────────────────
sample_sub_ids  = set(sample_sub['row_id'])
inferred_ids    = set(submission_df['row_id']) if len(submission_df) > 0 else set()
missing_ids     = sample_sub_ids - inferred_ids

if missing_ids:
    print(f'WARNING: {len(missing_ids)} row_ids missing from inference — filling with 0.5')
    filler_rows = [{'row_id': rid, **{s: 0.5 for s in SPECIES_IDS}} for rid in missing_ids]
    filler_df   = pd.DataFrame(filler_rows)
    submission_df = pd.concat([submission_df, filler_df], ignore_index=True)

# ── Reorder columns to exactly match sample_submission.csv ───────────
# Add any species columns missing from submission_df
for col in TARGET_COLUMNS:
    if col not in submission_df.columns:
        submission_df[col] = 0.5
submission_df = submission_df[TARGET_COLUMNS]

# ── Reorder rows to match sample_submission row order exactly ─────────
submission_df = sample_sub[['row_id']].merge(submission_df, on='row_id', how='left')
submission_df[SPECIES_IDS] = submission_df[SPECIES_IDS].fillna(0.5)

# ── Final assertions ─────────────────────────────────────────────────
assert list(submission_df.columns) == TARGET_COLUMNS, \
    f'FAILED: column order mismatch — got {list(submission_df.columns)[:5]}...'

assert len(submission_df) == len(sample_sub), \
    f'FAILED: row count mismatch: {len(submission_df)} vs {len(sample_sub)}'

assert submission_df.isnull().sum().sum() == 0, \
    f'FAILED: NaN values in submission ({submission_df.isnull().sum().sum()} cells)'

prob_vals = submission_df[SPECIES_IDS].values
assert (prob_vals >= 0).all(), \
    f'FAILED: negative values in submission (min={prob_vals.min():.4f})'
assert (prob_vals <= 1).all(), \
    f'FAILED: values > 1 in submission (max={prob_vals.max():.4f})'

# ── Write output ─────────────────────────────────────────────────────
submission_df.to_csv('submission.csv', index=False)
print(f'submission.csv written: {len(submission_df)} rows × {len(TARGET_COLUMNS)} columns')
print(f'  Prob stats: mean={prob_vals.mean():.4f} | '
      f'min={prob_vals.min():.4f} | max={prob_vals.max():.4f}')

top10 = pd.Series(prob_vals.mean(axis=0), index=SPECIES_IDS).nlargest(10)
print(f'Top-10 species by mean prediction:')
print(top10.to_string())

check_budget('CSV written')
total_elapsed = time.time() - START_TIME
print(f'\nTotal wall-clock time: {total_elapsed/60:.1f} min')
print('Done — submission.csv is ready.')